[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain学院

欢迎来到LangChain学院！

## 背景

在LangChain，我们致力于让构建LLM应用程序变得简单。您可以构建的一种LLM应用程序是智能代理（agent）。构建智能代理令人兴奋，因为它们可以自动化大量以前无法实现的任务。

但在实践中，构建能够可靠执行这些任务的系统极其困难。在与用户合作将代理投入生产的过程中，我们了解到通常需要更多的控制。您可能需要代理总是首先调用特定工具，或者根据其状态使用不同的提示。

为了解决这个问题，我们构建了[LangGraph](https://langchain-ai.github.io/langgraph/) —— 一个用于构建代理和多代理应用程序的框架。LangGraph与LangChain包分离，其核心设计理念是帮助开发者在代理工作流中添加更好的精确性和控制，适用于现实世界系统的复杂性。

## 课程结构

课程被结构化为一系列模块，每个模块专注于与LangGraph相关的特定主题。您会看到每个模块的文件夹，其中包含一系列笔记本。每个笔记本都会有一个视频来帮助讲解概念，但笔记本也是独立的，意味着它们包含解释，可以独立于视频查看。每个模块文件夹还包含一个`studio`文件夹，其中包含一组可以加载到[LangGraph Studio](https://github.com/langchain-ai/langgraph-studio)中的图，这是我们构建LangGraph应用程序的IDE。

## 设置

在开始之前，请按照`README`中的说明创建环境并安装依赖项。

## 聊天模型

在本课程中，我们将使用[聊天模型](https://python.langchain.com/v0.2/docs/concepts/#chat-models)，它们接受一系列消息作为输入并返回聊天消息作为输出。LangChain不托管任何聊天模型，而是依赖第三方集成。[这里](https://python.langchain.com/v0.2/docs/integrations/chat/)是LangChain中第三方聊天模型集成的列表！默认情况下，课程将使用[ChatOpenAI](https://python.langchain.com/v0.2/docs/integrations/chat/openai/)，因为它既受欢迎又性能良好。如前所述，请确保您有`OPENAI_API_KEY`。

让我们检查您的`OPENAI_API_KEY`是否已设置，如果没有，您将被要求输入它。

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community tavily-python

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

[这里](https://python.langchain.com/v0.2/docs/how_to/#chat-models)是一个有用的指南，介绍了您可以使用聊天模型做的所有事情，但我们将在下面展示一些亮点。如果您已经按照README中的说明运行了`pip install -r requirements.txt`，那么您已经安装了`langchain-openai`包。有了这个，我们可以实例化我们的`ChatOpenAI`模型对象。如果您是第一次注册API，您应该会收到可应用于任何模型的[免费积分](https://community.openai.com/t/understanding-api-limits-and-free-tier/498517)。您可以在[这里](https://openai.com/api/pricing/)查看各种模型的定价。笔记本将默认使用`gpt-4o`，因为它在质量、价格和速度方面有很好的平衡[详见此处](https://help.openai.com/en/articles/7102672-how-can-i-access-gpt-4-gpt-4-turbo-gpt-4o-and-gpt-4o-mini)，但您也可以选择价格较低的`gpt-3.5`系列模型。

有[一些标准参数](https://python.langchain.com/v0.2/docs/concepts/#chat-models)我们可以用聊天模型设置。最常见的两个是：

* `model`: 模型的名称
* `temperature`: 采样温度

`Temperature`控制模型输出的随机性或创造性，其中低温度（接近0）是更确定性和集中的输出。这适合需要准确性或事实性响应的任务。高温度（接近1）适合创意任务或生成多样化的响应。

In [3]:
from langchain_openai import ChatOpenAI
gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

LangChain中的聊天模型有许多[默认方法](https://python.langchain.com/v0.2/docs/concepts/#runnable-interface)。大部分情况下，我们将使用：

* `stream`: 流式返回响应块
* `invoke`: 在输入上调用链

而且，如前所述，聊天模型以[消息](https://python.langchain.com/v0.2/docs/concepts/#messages)作为输入。消息有一个角色（描述谁在说这个消息）和一个内容属性。我们稍后会详细讨论这个，但这里让我们只展示基础知识。

In [4]:
from langchain_core.messages import HumanMessage

# 创建一个消息
msg = HumanMessage(content="Hello world", name="Lance")

# 消息列表
messages = [msg]

# 用消息列表调用模型
gpt4o_chat.invoke(messages)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f33640a400', 'id': 'chatcmpl-CHixN8lnWyUyo6IYslHuXAYxsyLyz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--e26e08d6-5e2f-4720-89f8-3569c04b84e7-0', usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

我们得到一个`AIMessage`响应。另外，请注意我们可以直接用字符串调用聊天模型。当传入字符串作为输入时，它会被转换为`HumanMessage`，然后传递给底层模型。

In [5]:
gpt4o_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f33640a400', 'id': 'chatcmpl-CHixOIyJ69lasXkK0VAat60hWxYlb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d2280850-c2e0-481e-9be1-47da6e194913-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
gpt35_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CHixPRabBtNEVwM1G65Pa3AsB4thX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7ef51d62-3df0-441f-b956-1b6770661fc3-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

所有聊天模型的接口都是一致的，模型通常在每个笔记本开始时初始化一次。

因此，如果您对另一个提供商有强烈偏好，您可以轻松地在模型之间切换，而不需要更改下游代码。

## 搜索工具

您还会在README中看到[Tavily](https://tavily.com/)，这是一个为LLM和RAG优化的搜索引擎，旨在提供高效、快速和持久的搜索结果。如前所述，它很容易注册并提供慷慨的免费层。一些课程（在模块4中）将默认使用Tavily，但当然，如果您想为自己修改代码，也可以使用其他搜索工具。

In [7]:
_set_env("TAVILY_API_KEY")

TAVILY_API_KEY:  ········


In [8]:
from langchain_community.tools.tavily_search import TavilySearchResults
tavily_search = TavilySearchResults(max_results=3)
search_docs = tavily_search.invoke("What is LangGraph?")

/tmp/ipykernel_2854785/3227139208.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)


In [9]:
search_docs

[{'title': 'What is LangGraph? - Analytics Vidhya',
  'url': 'https://www.analyticsvidhya.com/blog/2024/07/langgraph-revolutionizing-ai-agent/',
  'content': 'To sum up, LangGraph is a major advancement in the development of AI agents. It enables developers to push the limits of what’s possible with AI agents by eliminating the shortcomings of earlier systems and offering a flexible, graph-based framework for agent construction and execution. LangGraph is positioned to influence the direction of artificial intelligence significantly in the future. [...] LangGraph is a library built on top of Langchain that is designed to facilitate the creation of cyclic graphs for large language model (LLM) – based AI agents.\n It views agent Objective Points about LangGraph and workflows as cyclic graph topologies, allowing for more variable and nuanced agent behaviors than linear execution models. [...] Frameworks such as LangGraph are becoming increasingly important as AI develops. LangGraph is mak